In [ ]:
# download_data.py
import time
import requests
import pandas as pd
from pathlib import Path

BASE_URL = "https://api.binance.com"

def fetch_klines_1m(symbol="BTCUSDT", minutes=20000, end_time_ms=None):
    if end_time_ms is None:
        end_time_ms = int(time.time() * 1000)

    klines = []
    cur_end = end_time_ms
    remaining = minutes

    while remaining > 0:
        limit = min(1000, remaining)
        r = requests.get(
            f"{BASE_URL}/api/v3/klines",
            params={"symbol": symbol, "interval": "1m", "limit": limit, "endTime": cur_end},
            timeout=10
        )
        r.raise_for_status()
        data = r.json()
        if not data:
            break

        klines = data + klines
        cur_end = data[0][0] - 1
        remaining -= len(data)
        time.sleep(0.5)

    df = pd.DataFrame(klines, columns=[
        "open_time","open","high","low","close","volume",
        "close_time","qav","num_trades","tbbv","tbqv","ignore"
    ])[["open_time","open","high","low","close","volume",
        "close_time","qav","num_trades","tbbv","tbqv"]]

    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["close"] = df["close"].astype(float)
    df = df.sort_values("open_time").reset_index(drop=True)
    return df

def download_and_cache(symbol="BTCUSDT", minutes=20000, cache_path="btc_1m.feather", force=False):
    p = Path(cache_path)
    df = fetch_klines_1m(symbol=symbol, minutes=minutes)
    df.to_feather(p)
    return df

In [2]:
df = download_and_cache(symbol="BTCUSDT", minutes=1000000, cache_path="btc_1m.feather")
print(df.head(), df.tail(), len(df))

Downloaded 1000 klines, remaining 999000 minutes
Downloaded 1000 klines, remaining 998000 minutes
Downloaded 1000 klines, remaining 997000 minutes
Downloaded 1000 klines, remaining 996000 minutes
Downloaded 1000 klines, remaining 995000 minutes
Downloaded 1000 klines, remaining 994000 minutes
Downloaded 1000 klines, remaining 993000 minutes
Downloaded 1000 klines, remaining 992000 minutes
Downloaded 1000 klines, remaining 991000 minutes
Downloaded 1000 klines, remaining 990000 minutes
Downloaded 1000 klines, remaining 989000 minutes
Downloaded 1000 klines, remaining 988000 minutes
Downloaded 1000 klines, remaining 987000 minutes
Downloaded 1000 klines, remaining 986000 minutes
Downloaded 1000 klines, remaining 985000 minutes
Downloaded 1000 klines, remaining 984000 minutes
Downloaded 1000 klines, remaining 983000 minutes
Downloaded 1000 klines, remaining 982000 minutes
Downloaded 1000 klines, remaining 981000 minutes
Downloaded 1000 klines, remaining 980000 minutes
Downloaded 1000 klin

In [5]:
df.memory_usage().sum() / 1024**2

np.float64(83.92346572875977)